In [1]:
from pyspark.sql import functions as F


In [2]:

# Load compacted Parquet
bronze = spark.read.parquet(
    "Files/raw/physionet_deterioration.parquet"
)

# Basic validation
row_count = bronze.count()
patient_count = bronze.select("patient_id").distinct().count()

print("Rows:", row_count)
print("Patients:", patient_count)
print("Columns:", len(bronze.columns))

print("\nColumns:")
print(bronze.columns)

print("\nTarget distribution:")
display(
    bronze.groupBy("SepsisLabel")
    .count()
    .withColumn(
        "percentage",
        F.round(F.col("count") / F.lit(row_count) * 100, 3)
    )
    .orderBy("SepsisLabel")
)

print("\nPatient-level deterioration distribution:")
display(
    bronze.groupBy("patient_id")
    .agg(F.max("SepsisLabel").alias("ever_positive"))
    .groupBy("ever_positive")
    .count()
    .orderBy("ever_positive")
)

# Save Bronze Delta table
(
    bronze.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("bronze_deterioration_hourly")
)

print("\nbronze_deterioration_hourly created successfully.")


Patient-level deterioration distribution:



bronze_deterioration_hourly created successfully.
